# 进阶:把城市变成 3D

前面贴了角色、按情景调了高度。这本把占地挤成 3D 量体,三种方式看:matplotlib(页面内、程式码定角度)、plotly(浏览器滑鼠转)、OBJ(汇出给 Rhino/Blender)。

## 怎么执行

- 点一格,按 **Shift+Enter** 执行;或选单 Run → Run All Cells 从头跑。
- 结果与图会显示在该格下面。
- 跟着说明,在标 ✏️ 的地方改设定,再重跑那一步。

In [ ]:
# 讓練習本找到 engine 裡的程式碼(這格不用改,直接執行)
import sys, os
for _p in (".", "engine", os.path.join("engine", "steps")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

## 准备

汇入工具、让图显示在页面,并取预设大巴窑建筑、贴好角色标签。

In [ ]:
import common, plots
import matplotlib; matplotlib.use("module://matplotlib_inline.backend_inline")
%matplotlib inline
df = common.assign_all(common.current_buildings())   # 載入並貼好角色標籤
print("建築數:", len(df), "|", df["stakeholder"].value_counts().to_dict())
print("本次輸出資料夾:", common.OUT.relative_to(common.ROOT), "(每次跑進各自時間戳夾,不覆寫)")
print(common.honest_note())

## 选用哪个高度

可用 current(原始观测高),或某权力情景的高度。改 `HEIGHT_SCEN` 切换。

In [ ]:
HEIGHT_SCEN = "current"   # ✏️ 試 "developer_led"/"community_led"/"state_eco"/"developer_renewal"
scen = common.load_scenarios()
assert HEIGHT_SCEN in scen, "沒有這個情景,可選:%s" % list(scen.keys())
df["h3d"] = df["height_m"].values if HEIGHT_SCEN == "current" else common.scenario_heights(df, scen[HEIGHT_SCEN]).values
print("3D 用高度:", HEIGHT_SCEN, "| 平均 %.1f m 最高 %.1f m" % (df.h3d.mean(), df.h3d.max()))

## 取多少栋

为了画得快,只取占地最大的约 200 栋。改 `TOP_N` 调整,越大越慢。

In [ ]:
TOP_N = 200   # ✏️ 取佔地最大的幾棟;越大越慢
sub = df.sort_values("area_m2", ascending=False).head(TOP_N).copy()
print("3D 子集:%d 棟,佔總佔地 %.0f%%" % (len(sub), 100*sub.area_m2.sum()/df.area_m2.sum()))

## 方式 A:matplotlib

把每栋占地按高度挤成盒子,在页面显示。角度由 elev(俯仰)、azim(旋转)控制。

In [ ]:
plots.city_3d(sub, height_col="h3d", elev=30, azim=-60)   # ✏️ 改 elev/azim 換視角

## 方式 B:plotly

生成可在浏览器用滑鼠拖动旋转的 3D,存成 html。转到满意角度后,用工具列相机钮截图。

In [ ]:
fig = plots.city_3d_plotly(sub, height_col="h3d")         # 可在瀏覽器拖動旋轉
out_html = common.OUT / ("city3d_%s.html" % HEIGHT_SCEN)
fig.write_html(str(out_html)); print("→ 寫好可旋轉 3D:", out_html.relative_to(common.ROOT))
fig.show()

## 方式 C:汇出 OBJ

用真实占地挤成量体,汇出 .obj 给 Rhino 或 Blender 精确建模量测。

In [ ]:
obj, nv, nf = common.extrude_obj(sub, height_col="h3d")   # 匯出 OBJ 給 Rhino/Blender
out_obj = common.OUT / ("city_%s.obj" % HEIGHT_SCEN)
out_obj.write_text(obj, encoding="utf-8")
print("→ 匯出 OBJ:", out_obj.relative_to(common.ROOT), "| 頂點%d 面%d" % (nv, nf))

## 小结

- matplotlib:程式码定角度,适合批量固定视角。
- plotly:滑鼠找角度,适合挑视角截图。
- OBJ:交专业软体,适合精确建模。